In [61]:
from openai import OpenAI


import pandas as pd
import numpy as np
import re
import json


In [62]:
df = pd.read_csv('../../data/train/cleaned_train.csv')
print(f"Total records: {len(df)}")


Total records: 14603


# **Análisis de descripción**

### Objetivo: 
Analizar la descripción de los autos para obtener información del estado (si está roto, necesita repuestos, etc.)

### Método:
 - Elegir descripciónes de autos usados (> 2000km), algunos que estén en buen estado y otros no. 
 - Usar ChatGPT 4.1 Nano para analizar la descripción y obtener un score de 0 a 100. (Usar batch request para analizar varias descripciones a la vez y abaratar costos)










In [63]:
# Filter by kilometers > 5000
df_filtered = df[df['Kilómetros'].str.lower()
    .str.replace('km', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
    .replace('', np.nan)
    .astype(float) > 5000].copy()

print(f"Records with > 2000km: {len(df_filtered)}")

# Define damage-related keywords
damage_keywords = [
    'rot', 'a reparar', 'detalle', 'paragolpes', 'reparar', 'raya',
    'golpe', 'abolladura', 'daño', 'falla', 'problema', 'averia',
    'desperfecto', 'mal estado', 'necesita', 'arreglar', 'bollo', 'choc',
    'cambiar', 'arreglar'
]

def contains_damage_keywords(text):
    if pd.isna(text):
        return False
    text_lower = str(text).lower()
    return any(keyword in text_lower for keyword in damage_keywords)

# Apply filter for descriptions containing damage keywords
df_damaged = df_filtered[df_filtered['Descripción'].apply(contains_damage_keywords)].copy()
print(f"Records with damage keywords in description: {len(df_damaged)}")

print(f"\nFinal filtered dataset: {len(df_damaged)} records")


Records with > 2000km: 10334
Records with damage keywords in description: 1269

Final filtered dataset: 1269 records


In [64]:
print("Random sample of descriptions:")
print("=" * 80)

# Get 10 random rows
random_rows = df_damaged.sample(n=10)

for idx, row in random_rows.iterrows():
    print(f"\nDescripción: {row['Descripción']}")
    print("-" * 40)

Random sample of descriptions:

Descripción: CONTADO U$S 9.500-------------------------------NO FINANCIABLE-------------------------------ECOSPORT 2008 ¡¡¡PRIMERA MANO DE 0KM!!!AÑO 2008 || MOTOR 1.6 CADENERO || 146.000 KM REALES COMPROBABLESAIRE, DOBLE AIRBAG, DIRECCION, CIERRE CENTRALIZADO, ALARMAEXCELENTE ESTADO !! SIN DETALLES.¡¡TOMAMOS TU AUTO EN PARTE DE PAGO!!VTV VIGENTE, GRABADO DE AUTOPARTES Y LIBRE DE TODO TIPO DE DEUDAS.¡¡ TENEMOS GESTORIA PROPIA PARA LA TRANSFERENCIA DEL VEHÍCULO !!HORARIO DE ATENCIÓN: LUNES A VIERNES 9.30 A 17.45hsSabados de 9.30 a 12.45hsTURDERA, LOMAS DE ZAMORA
----------------------------------------

Descripción: Con detalles de carrocería evidenciados en las fotos.
----------------------------------------

Descripción: FORD ECOSPORT TITANIUMAÑO 2018KILÓMETROS 75.000MOTOR 1.5 NAFTACAJA MANUALTECHO CORREDIZOTAPIZADO CUERO CAMELCÁMARA RETROCESOSomos WEST MOTORS, una empresa dedicada a la comercializacion de vehiculos usados y 0km para brindar una operacio

# Usar modelo para analizar la descripción

Leer documentación de [OpenAI](https://platform.openai.com/docs/api-reference/batch/create) para Batch requests.


In [65]:
with open('prompt.txt', 'r', encoding='utf-8') as f:
    system_prompt = f.read().strip()

jsonl_lines = []

for idx, row in df_damaged.iterrows():
    description = row['Descripción']
    
    if pd.isna(description) or str(description).strip() == '':
        continue
    
    request = {
        "custom_id": f"request-{idx}",
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": {
            "model": "gpt-4.1-nano",
            "messages": [
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": f"{description}"
                }
            ],
            "max_tokens": 500,
            "response_format": {"type": "json_object"}  # Ensure JSON response
        }
    }
    
    jsonl_lines.append(json.dumps(request, ensure_ascii=False))

with open('batch_requests.jsonl', 'w', encoding='utf-8') as f:
    for line in jsonl_lines:
        f.write(line + '\n')

print(f"Created batch_requests.jsonl with {len(jsonl_lines)} requests")
print(f"Sample request structure:")
print(json.dumps(json.loads(jsonl_lines[0]), indent=2, ensure_ascii=False))

Created batch_requests.jsonl with 1269 requests
Sample request structure:
{
  "custom_id": "request-17",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-4.1-nano",
    "messages": [
      {
        "role": "system",
        "content": "Dado el texto de una publicación de venta de un vehículo, necesito que evalúes y categorices los daños mencionados en la descripción del vehículo según las siguientes categorías:\n\n\"cosmetic_damage\": 1 si menciona daños cosméticos como rayones, abolladuras menores, pintura deteriorada, detalles en tapizados o elementos visuales, 0 si no.\n\n\"functional_damage\": 1 si menciona daños funcionales o problemas mecánicos menores pero el auto aún funciona normalmente (por ejemplo: suspensión desgastada, desgaste de frenos), 0 si no.\n\n\"severe_damage\": 1 si menciona daños severos en componentes críticos como motor, transmisión, caja de cambios o sistema eléctrico que afectan gravemente la operación del vehículo, 0 si no.

### Subir batch

In [66]:
client = OpenAI()

batch_input_file = client.files.create(
    file=open("batch_requests.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-FXPzSkMjorPEzVB9yrmsJQ', bytes=2785421, created_at=1750198126, filename='batch_requests.jsonl', object='file', purpose='batch', status='processed', expires_at=None, status_details=None)


### Crear batch

In [67]:
batch_input_file_id = batch_input_file.id
new_batch = client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={
        "description": "Primer batch con 900 datos, filtrados por kilometraje y si contiene daños en la descripción"
    }
)

In [73]:
batch = client.batches.retrieve(new_batch.id)

print(batch.to_json())

file_response = client.files.content(batch.output_file_id)


{
  "id": "batch_6851e76f9c848190ad86aa941b3c0d50",
  "completion_window": "24h",
  "created_at": 1750198127,
  "endpoint": "/v1/chat/completions",
  "input_file_id": "file-FXPzSkMjorPEzVB9yrmsJQ",
  "object": "batch",
  "status": "completed",
  "cancelled_at": null,
  "cancelling_at": null,
  "completed_at": 1750198554,
  "error_file_id": null,
  "errors": null,
  "expired_at": null,
  "expires_at": 1750284527,
  "failed_at": null,
  "finalizing_at": 1750198469,
  "in_progress_at": 1750198129,
  "metadata": {
    "description": "Primer batch con 900 datos, filtrados por kilometraje y si contiene daños en la descripción"
  },
  "output_file_id": "file-PoJiFKWQNtLBJBn7XJsMX8",
  "request_counts": {
    "completed": 1269,
    "failed": 0,
    "total": 1269
  }
}


In [75]:
# Parse the batch response
results = []
for line in file_response.text.strip().split('\n'):
    if line.strip():
        try:
            data = json.loads(line)
            custom_id = data['custom_id']
            response_body = data['response']['body']
            
            # Extract the content from the assistant's message
            content = response_body['choices'][0]['message']['content']
            
            # Parse the JSON content
            result_data = json.loads(content)
            
            # Extract the original row index from custom_id
            row_idx = int(custom_id.replace('request-', ''))
            
            results.append({
                'original_idx': row_idx,
                'cosmetic_damage': result_data['cosmetic_damage'],
                'functional_damage': result_data['functional_damage'],
                'severe_damage': result_data['severe_damage']
            })
            
        except Exception as e:
            print(f"Error parsing line: {e}")
            print(f"Line content: {line[:200]}...")

# Create DataFrame with results
results_df = pd.DataFrame(results)
print(f"Parsed {len(results_df)} results")

# Reset index of original data to get the original row numbers
df_final_reset = df_damaged.reset_index()

# Merge with original filtered data
final_df = df_final_reset.merge(
    results_df, 
    left_on='index',
    right_on='original_idx', 
    how='left'
)

output_df = final_df[['index', 'Kilómetros', 'Descripción', 'cosmetic_damage', 'functional_damage', 'severe_damage']].copy()
output_df = output_df.rename(columns={'index': 'idx'})

output_df.to_csv('car_damage_analysis.csv', index=False)
print(f"Saved results to car_damage_analysis.csv")
print(f"Shape: {output_df.shape}")

print("\nDamage Summary:")
print(f"Cars with cosmetic damage: {output_df['cosmetic_damage'].sum()}")
print(f"Cars with functional damage: {output_df['functional_damage'].sum()}")
print(f"Cars with severe damage: {output_df['severe_damage'].sum()}")

print(f"\nPercentage breakdown:")
total_cars = len(output_df)
print(f"Cosmetic damage: {(output_df['cosmetic_damage'].sum() / total_cars * 100):.1f}%")
print(f"Functional damage: {(output_df['functional_damage'].sum() / total_cars * 100):.1f}%")
print(f"Severe damage: {(output_df['severe_damage'].sum() / total_cars * 100):.1f}%")

any_damage = (output_df['cosmetic_damage'] | output_df['functional_damage'] | output_df['severe_damage']).sum()
print(f"\nCars with any type of damage: {any_damage} ({(any_damage / total_cars * 100):.1f}%)")

Parsed 1269 results
Saved results to car_damage_analysis.csv
Shape: (1269, 6)

Damage Summary:
Cars with cosmetic damage: 131
Cars with functional damage: 17
Cars with severe damage: 8

Percentage breakdown:
Cosmetic damage: 10.3%
Functional damage: 1.3%
Severe damage: 0.6%

Cars with any type of damage: 148 (11.7%)


In [76]:
# Analyze damage combinations
print("Damage combination analysis:")
print("=" * 50)

# Create damage combination column
output_df['damage_combination'] = output_df.apply(
    lambda row: f"C:{row['cosmetic_damage']} F:{row['functional_damage']} S:{row['severe_damage']}", 
    axis=1
)

# Count damage combinations
damage_combinations = output_df['damage_combination'].value_counts()
print("Damage combinations:")
for combo, count in damage_combinations.items():
    percentage = (count / len(output_df)) * 100
    print(f"{combo}: {count} cars ({percentage:.1f}%)")

# Show examples of cars with different damage types
print(f"\nExamples of cars with cosmetic damage:")
cosmetic_damage_cars = output_df[output_df['cosmetic_damage'] == 1].head(3)
for _, row in cosmetic_damage_cars.iterrows():
    print(f"\nCar {row['idx']}:")
    print(f"Description: {row['Descripción'][:200]}...")

print(f"\nExamples of cars with functional damage:")
functional_damage_cars = output_df[output_df['functional_damage'] == 1].head(3)
for _, row in functional_damage_cars.iterrows():
    print(f"\nCar {row['idx']}:")
    print(f"Description: {row['Descripción'][:200]}...")

print(f"\nExamples of cars with severe damage:")
severe_damage_cars = output_df[output_df['severe_damage'] == 1].head(3)
for _, row in severe_damage_cars.iterrows():
    print(f"\nCar {row['idx']}:")
    print(f"Description: {row['Descripción'][:200]}...")

Damage combination analysis:
Damage combinations:
C:0 F:0 S:0: 1121 cars (88.3%)
C:1 F:0 S:0: 127 cars (10.0%)
C:0 F:1 S:0: 9 cars (0.7%)
C:0 F:1 S:1: 4 cars (0.3%)
C:0 F:0 S:1: 4 cars (0.3%)
C:1 F:1 S:0: 4 cars (0.3%)

Examples of cars with cosmetic damage:

Car 353:
Description: Vendo Glk 300 city Services ofíciales hasta el 2024 (Panamer).A los 185.000 se le cambiaron los 4 amortiguadores. A los 197.000 se le hizo un mantenimiento muy grande más de 10.000 dólares:-cambio de ...

Car 361:
Description: Vendo nuestro auto – Segundo dueño directoLo compramos con 80.000 km y actualmente tiene 102.000 km.VTV al díaLas 2 llaver orginalesAuxilio con cero uso.Libre de multasEscaneable sin problemasInterior...

Car 366:
Description: Ssangyong Musso motor Mercedes Benz 601 2.3 TD . Motor en excelentes condiciones . Turbo reparado a nuevo . Casco entero , sin golpes , solo algunas raspaduras . Solo esta rota ( como se ve en la foto...

Examples of cars with functional damage:

Car 2156:
Descrip